# 05 — Real data walkthrough (your actual `.mat` files)

End-to-end on the actual study data. Uses the corrected loader from `python/io.py` which:

- anchors site-code matching on the strict slot `-<CODE>_` (no false positives from leading sessionTags like `4088-SNB-PRO_..._Prolific_...`),
- excludes `-original.` legacy backups,
- filters single-ISI vs `_Multi-p` (multi-ISI) sessions,
- computes d′ at ISI=0 on the fly (the `dprime_ISI0` field is not stored in the `.mat` files).

Switch `CONDITION` and `IS_MULTI_ISI` at the top to look at a different cut.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd(); ROOT = HERE.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from python import (
    load_group,
    bootstrap_intergroup_correlation_sem,
    paired_bootstrap_compare_correlations,
)

In [ ]:
BASE_DIR = ROOT.parents[2] / 'Data' / 'RecognitionMemory' / 'Results'
assert BASE_DIR.exists(), f'Data directory not found: {BASE_DIR}'

CONDITION = 'Globalized-Music'   # or 'Industrial-Nature'
IS_MULTI_ISI = False             # single-ISI sessions only (matches MATLAB driver default)
MIN_ISI0_DPRIME = 2.0
N_SPLITS = 1000                  # bump to 10_000 for final
N_BOOT = 1000                    # bump to 5000 for final
N_BOOT_PAIRED = 2000
MIN_RESP = 2

SITES = {
    'US'       : ('PRO', 'BOS', 'CAM'),
    'SanBorja' : ('SBO', 'SNB', 'SBJ'),
    'Tsimane'  : ('NVM', 'MAJ', 'MAN', 'NUM', 'NUV', 'CVR'),
}
print(f'Loading {CONDITION}  |  multi-ISI={IS_MULTI_ISI}  |  d_prime >= {MIN_ISI0_DPRIME}')

## Load each group

`load_group` lists matching .mat files, builds hit/FA matrices, and computes within-group split-half reliability.

In [ ]:
rng = np.random.default_rng(0)
groups = {}
for label, codes in SITES.items():
    print(f'\n[{label}]  codes={codes}')
    groups[label] = load_group(
        BASE_DIR, codes, CONDITION,
        min_isi0_dprime=MIN_ISI0_DPRIME, is_multi_isi=IS_MULTI_ISI,
        n_splits=N_SPLITS, split_dim=1, corr_type='Spearman',
        rng=rng,
    )

In [ ]:
import pandas as pd
rows = []
for label, g in groups.items():
    if g is None:
        rows.append([label, 0, 0, None, None]); continue
    rows.append([label, g.n_subjects, len(g.items),
                 round(g.sb_hit, 3), round(g.sb_fa, 3)])
pd.DataFrame(rows, columns=['group','n_subj','n_items','SB_hit','SB_fa'])

## Intergroup bootstraps (each pair)

Headline = sample point r; CI = bootstrap percentile; corrected = attenuation correction with mode='fixed'.

In [ ]:
TRIAL = 'hit'
pairs = [('US','SanBorja'), ('US','Tsimane'), ('SanBorja','Tsimane')]
boot_results = {}
for a, b in pairs:
    print(f'\n--- r({a}, {b}) ---')
    boot_results[(a, b)] = bootstrap_intergroup_correlation_sem(
        groups[a], groups[b], trial_type=TRIAL,
        n_boot=N_BOOT, min_resp=MIN_RESP,
        reliability_mode='fixed', correct_atten=True,
        rng=np.random.default_rng(1),
    )

In [ ]:
rows = []
for (a, b), r in boot_results.items():
    rows.append([f'{a}-{b}', round(r.point_raw, 3),
                 f'[{r.ci_raw[0]:.3f}, {r.ci_raw[1]:.3f}]',
                 round(r.point_corr, 3),
                 f'[{r.ci_corr[0]:.3f}, {r.ci_corr[1]:.3f}]',
                 r.point_items_n])
pd.DataFrame(rows, columns=['pair','r_raw','95% CI raw','r_corr','95% CI corr','n_items'])

## Paired-bootstrap comparison

The substantive test: does r(US, SanBorja) differ from r(US, Tsimane)? Etc. Reports both p-value variants — if they diverge, trust the recentered-null and treat the divergence as a small-N signal.

In [ ]:
paired = paired_bootstrap_compare_correlations(
    groups['US'], groups['SanBorja'], groups['Tsimane'],
    trial_type=TRIAL,
    n_boot=N_BOOT_PAIRED, use_spearman=True,
    bootstrap_dim=1, min_resp=MIN_RESP,
    rng=np.random.default_rng(2),
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, key, title in zip(axes,
        ['AB_minus_AC','AB_minus_BC','AC_minus_BC'],
        ['r(US,SB) - r(US,Tsi)','r(US,SB) - r(SB,Tsi)','r(US,Tsi) - r(SB,Tsi)']):
    d = paired.diffs[key]
    ax.hist(d, bins=40, color='#4060b0', alpha=0.7)
    ax.axvline(0, color='k', lw=1)
    ax.axvline(paired.observed_diffs[key], color='r', lw=1.5, ls='--', label='observed')
    ax.set_title(title); ax.set_xlabel('diff'); ax.legend()
axes[0].set_ylabel('count')
plt.suptitle(f'{CONDITION} | {TRIAL}: paired-bootstrap diff distributions', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
rows = []
for key, label in [('AB_minus_AC','US-SB vs US-Tsi'),
                    ('AB_minus_BC','US-SB vs SB-Tsi'),
                    ('AC_minus_BC','US-Tsi vs SB-Tsi')]:
    rows.append([label,
                 round(paired.observed_diffs[key], 3),
                 f'[{paired.ci[key][0]:.3f}, {paired.ci[key][1]:.3f}]',
                 round({
                     'AB_minus_AC': paired.ab_vs_ac,
                     'AB_minus_BC': paired.ab_vs_bc,
                     'AC_minus_BC': paired.ac_vs_bc,
                 }[key], 4),
                 round(paired.straddle[
                     {'AB_minus_AC':'ab_vs_ac','AB_minus_BC':'ab_vs_bc','AC_minus_BC':'ac_vs_bc'}[key]
                 ], 4)])
pd.DataFrame(rows, columns=['comparison','observed diff','95% CI','p (recentered-null)','p (straddle-zero)'])

**Reading the result.**

- The **observed diff** column is the difference between the two sample correlations.
- The **CI** column is the 95% bootstrap percentile interval around that diff. If it covers 0, the diff is not distinguishable from chance under the bootstrap.
- The **p (recentered-null)** column is the test we recommend reporting.
- The **p (straddle-zero)** column is the legacy formulation. If it's much larger than the recentered-null p, that signals skew in the bootstrap diff distribution at your N — the recentered-null is more honest in that case.

Before reporting any of these as significant, run notebook 04 with the Ns from the table above to confirm the test is calibrated at your sample size.